# Consumer Discretionary, point-in-time GICS sample: FinBERT for the added calls

Scores only the 595 calls that the point-in-time GICS sample adds (plus 20 already-scored calls as a check). T4: ~20 min.

1. Runtime > Change runtime type > **GPU (T4)**.
2. Put the `cd_pit_finbert_input` folder (unzipped) or the zip anywhere in My Drive (licensed WRDS text: keep private, delete afterwards).
3. Run all. Download `MyDrive/cd_finbert/cd_pit_finbert_output.zip`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install transformers pyarrow
# Find the inputs anywhere in My Drive: the zip, or the unzipped folder, and copy them to /content/cdpit
import glob, os, shutil, zipfile
W = "/content/cdpit"
shutil.rmtree(W, ignore_errors=True)
os.makedirs("/content/drive/MyDrive/cd_finbert", exist_ok=True)
zips = glob.glob("/content/drive/MyDrive/**/cd_pit_finbert_input*.zip", recursive=True)
dirs = [os.path.dirname(s) for s in glob.glob("/content/drive/MyDrive/**/finbert_score_calls.py", recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(s), "ids.csv"))]
if dirs:
    print("using folder:", dirs[0])
    shutil.copytree(dirs[0], W)
elif zips:
    print("using zip:", zips[0])
    zipfile.ZipFile(zips[0]).extractall(W)
else:
    raise FileNotFoundError("no folder with finbert_score_calls.py + ids.csv (or cd_pit_finbert_input.zip) in My Drive; "
                            "a fresh upload can take a minute to appear")
if not os.path.exists(f"{W}/finbert_score_calls.py"):   # zip re-made by hand with an extra top folder
    inner = [d for d in glob.glob(f"{W}/*/") if os.path.exists(d + "finbert_score_calls.py")]
    assert inner, f"finbert_score_calls.py not found; got {os.listdir(W)}"
    for x in os.listdir(inner[0]):
        shutil.move(inner[0] + x, W)
for need in ["finbert_score_calls.py", "it_finbert_features.py", "ids.csv"]:
    assert os.path.exists(f"{W}/{need}"), f"missing {need}"
n = len(glob.glob(f"{W}/text/text_*.parquet"))
assert n == 15, f"expected 15 text files in text/, found {n} (did the whole folder finish uploading?)"
print("inputs OK:", sorted(os.listdir(W)), n, "text files")

In [ ]:
!cd /content/cdpit && python finbert_score_calls.py --text-dir text --ids ids.csv --out fb_pit_new.parquet --batch-size 128 2>&1 | tee fb_pit_run.log
import os
assert os.path.exists("/content/cdpit/fb_pit_new.parquet"), "scoring failed: read the log above"
import pandas as pd
f = pd.read_parquet("/content/cdpit/fb_pit_new.parquet"); print(f.shape); print(f.describe().T)

In [ ]:
!cd /content/cdpit && rm -f /content/drive/MyDrive/cd_finbert/cd_pit_finbert_output.zip && zip -q /content/drive/MyDrive/cd_finbert/cd_pit_finbert_output.zip fb_pit_new.parquet fb_pit_run.log
!unzip -l /content/drive/MyDrive/cd_finbert/cd_pit_finbert_output.zip